In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from matplotlib import pyplot as plt

data = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')

In [ ]:
data.head()

In [ ]:
data = np.array(data)
np.random.shuffle(data)

data = data.T

Y = data[0]
X = data[1:] / 255.0

m = Y.size

X_train = X[:, 1000:m]
Y_train = Y[1000:m]

X_dev = X[:, :1000]
Y_dev = Y[:1000]

Splitting data into a training set and dev set to allow for cross validation and prevent overfitting

In [ ]:
def init_param():
    W1 = np.random.randn(10,784) * np.sqrt(2 / 784)
    b1 = np.zeros((10,1))
    W2 = np.random.randn(10,10) * np.sqrt(2 / 10)
    b2 = np.zeros((10,1))
    
    return W1,b1, W2, b2

def ReLU(Z):
    return np.maximum(0,Z)

def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    exp = np.exp(Z)
    return exp / np.sum(exp, axis=0, keepdims=True)

def deriv_ReLU(Z):
    return Z > 0

def forward_prop(W1,b1,W2,b2,X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1,A1,Z2,A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size,Y.max() + 1))
    one_hot_Y[np.arange(Y.size),Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def back_prop(Z1,A1,Z2,A2,W2,X,Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [ ]:
def get_pred(A2):
    return np.argmax(A2, 0)

def get_acc(preds, Y):
    print(preds, Y)
    return np.sum(preds == Y)/ Y.size

def gd(X,Y,its,alpha):
    W1,b1,W2,b2 = init_param()
    for i in range(its):
        Z1, A1, Z2, A2 = forward_prop(W1,b1,W2,b2,X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1,b1,W2,b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if (i%5 == 0):
            print("iteration: ", i)
            print("accuracy: ", get_acc(get_pred(A2),Y))
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gd(X_train, Y_train, 500, 0.1)